# Understand embeddings with Word2Vec

### Exercise objectives:
- Convert 🔠 words to 🔢 vector representations thanks to embeddings
- Discover the powerful Word2Vec algorithm

<hr>

_Embeddings_ are representations of words using vectors. These embeddings can be learned within a Neural Network. But it can take time to converge. Another option is to learn them as a first step. Then, use them directly to feed the word representations into a Recurrent Neural Network. 

▶️ Run this cell and make sure the version of 📚 [Gensim - Word2Vec](https://radimrehurek.com/gensim/auto_examples/index.html) you are using is ≥ 4.0!

In [2]:
!pip freeze | grep gensim

gensim==4.3.3


In [3]:
!pip freeze | grep tensorflow

tensorflow-cpu==2.10.0
tensorflow-datasets==4.6.0
tensorflow-estimator==2.10.0
tensorflow-io-gcs-filesystem==0.27.0
tensorflow-metadata==1.10.0


# The data

Keras provides many datasets, among which is the IMDB dataset 🎬:
- It is comprised of sentences that are ***movie reviews***. 
- Each of these reviews is related to a score given by the reviewer.

❓ **Question** ❓ First of all, let's load the data. You don't have to understand what is going on in the function, it does not matter here.

⚠️ **Warning** ⚠️ The `load_data` function has a `percentage_of_sentences` argument. Depending on your computer, there are chances that too many sentences will make your compute slow down, or even freeze - your RAM can overflow. For that reason, **you should start with 10% of the sentences** and see if your computer can handle it. Otherwise, rerun with a lower number.  

⚠️ **DISCLAIMER** ⚠️ **No need to play _who has the biggest_ (RAM) !** The idea is to get to run your models quickly to prototype. Even in real life, it is recommended that you start with a subset of your data to loop and debug quickly. So increase the number only if you are into getting the best accuracy. 

In [4]:
###########################################
### Just run this cell to load the data ###
###########################################

import tensorflow_datasets as tfds
from tensorflow.keras.preprocessing.text import text_to_word_sequence

def load_data(percentage_of_sentences=None):
    train_data, test_data = tfds.load(name="imdb_reviews", split=["train", "test"], batch_size=-1, as_supervised=True)

    train_sentences, y_train = tfds.as_numpy(train_data)
    test_sentences, y_test = tfds.as_numpy(test_data)

    # Take only a given percentage of the entire data
    if percentage_of_sentences is not None:
        assert(percentage_of_sentences> 0 and percentage_of_sentences<=100)

        len_train = int(percentage_of_sentences/100*len(train_sentences))
        train_sentences, y_train = train_sentences[:len_train], y_train[:len_train]

        len_test = int(percentage_of_sentences/100*len(test_sentences))
        test_sentences, y_test = test_sentences[:len_test], y_test[:len_test]

    X_train = [text_to_word_sequence(_.decode("utf-8")) for _ in train_sentences]
    X_test = [text_to_word_sequence(_.decode("utf-8")) for _ in test_sentences]

    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = load_data(percentage_of_sentences=10)

2025-04-10 15:02:49.412950: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


<b><u>Embeddings in the previous challenge</u></b>:

In the previous exercise, we jointly learned a representation for the words, and fed this representation to a RNN, as shown down below 👇: 

<img src="layers_embedding.png" width="400px" />

However, this increases the number of parameters to learn, which slows down and increases the difficulty of convergence!

<b><u>Embeddings in the current challenge</u></b>:

For this reason, we will separate the steps of learning the word representation and feeding it into a RNN. As shown here: 

<img src="word2vec_representation.png" width="400px" />

We will learn the embedding with Word2Vec.

The drawback is indeed that the learned embeddings are not _specifically_ designed for our task. However, learning them independently of the task at hand (sentiment analysis) has some advantages: 
- it is very fast to do in general (with Word2Vec)
- the representation learned by Word2Vec is still meaningful 
- the convergence of the RNN alone will be easier and faster

So let's learn an embedding with Word2Vec and see how meaningful it is!

# Embedding with Word2Vec

Let's use Word2Vec to embed the words of our sentences. Word2Vec will be able to convert each word to a fixed-size vectorial representation.

For instance, we will have:
- 🐶 _dog_ $\rightarrow$ [0.1, -0.3, 0.8]
- 🐱 _cat_ $\rightarrow$ [-1.1, 2.3, 0.7]
- 🍏 _apple_ $\rightarrow$ [3.1, 0.9, -4.7]

Here, your embedding space is of size 3.

***What is a "good" numerical representation of words?***

- ***Words with close meanings should be geometrically close in your embedding space!***

    - Look at the following example which represents a bi-dimensional embedding space.

![Embedding](word_embedding.png)

❓ **Question** ❓ Let's run Word2Vec! 

[📚 **Gensim**](https://radimrehurek.com/gensim/)  is a great Python package that makes the use of the Word2Vec algorithm easy to implement, fast and accurate (which is not an easy task!).

1. The following code imports Word2Vec from Gensim. 

2. The second line learns the embedding representation of the words thanks to the sentences in `X_train`. 
3. The third line stores the words and their trained embeddings in `wv`.

In [5]:
from gensim.models import Word2Vec
word2vec = Word2Vec(sentences=X_train)

wv = word2vec.wv

Let's look at the embedded representation of some words.

You can use `wv` as a dictionary.
For instance, `wv['dog']` will return a representation of `dog` in the embedding space.

❓ **Question** ❓ Try different words - especially, try non-existing words to see that they don't have any representation (which is perfectly normal as their representation was not learned). 

In [6]:
# YOUR CODE HERE
# Try existing words
print(wv['dog'])
print(wv['cat'])
print(wv['movie'])
print(wv['action'])

# Try non-existing words
try:
    print(wv['laaaaaaaaaame'])
except KeyError as e:
    print(f"Error: {e}")

[-0.22714745  0.19405112 -0.1386999   0.24941334 -0.04908781 -0.3311056
  0.04300616  0.54521394 -0.20382278 -0.17492259 -0.09856719 -0.32534418
  0.00509208  0.13393596 -0.07510212 -0.23136133  0.20640968 -0.28244385
  0.01001796 -0.3503633   0.15261197  0.05287292  0.2071617  -0.06681068
  0.02715622  0.01026951 -0.21772675 -0.14462821 -0.16581975  0.03435599
  0.16129416  0.04438954  0.14132668 -0.36810863 -0.1254373   0.2799741
  0.07292115 -0.16979854 -0.28132215 -0.34247726 -0.25023475 -0.2746554
 -0.21606311  0.2629408   0.33695105 -0.04322379 -0.2068124  -0.08297637
  0.29486018  0.16783881  0.11493675 -0.1266197  -0.21634606  0.00489698
 -0.20001711  0.15413477  0.14778507 -0.12024778 -0.28609574  0.03304134
  0.11608295  0.01688347 -0.00797202  0.1411709  -0.26147443  0.21055087
 -0.05512521  0.17046666 -0.30610996  0.34905156 -0.16581985  0.208298
  0.30846512 -0.21867412  0.38072476  0.1643347   0.03871876  0.0960714
 -0.17894976 -0.0427779  -0.25264263 -0.14941606 -0.25693

❓ **Question** ❓ What is the size of each word representation, and therefore, what is the size of the embedding space?

In [7]:
# YOUR CODE HERE
# The size of each word representation is the vector size of the embedding
embedding_size = len(wv['movie'])
print(f"The size of each word representation (embedding space) is: {embedding_size}")

The size of each word representation (embedding space) is: 100


🧐 How do we know whether this embedding make any sense or not? 

💡 To investigate this question, we will check that words with a close meaning have close representations. 

👉 Let's use the [**`Word2Vec.wv.most_similar`**](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.most_similar) method that, given an input word, displays the "closest" words in the embedding space. If the embedding is well done, then words with similar meanings will have similar representation in the embedding space.

❓ **Question** ❓ Try out the `most_similar` method on different words. 

🧑🏿‍🏫 The quality of the closeness will depend on the quality of your embedding, and thus, depend on the number of sentences that you have loaded and from which you create your embedding.

In [8]:
# Try the most_similar method on different words
print(wv.most_similar('movie'))
print(wv.most_similar('action'))
print(wv.most_similar('good'))
print(wv.most_similar('bad'))

[('film', 0.9594041109085083), ('show', 0.8480587005615234), ('thing', 0.8151875138282776), ('sequel', 0.8037909269332886), ('ending', 0.7763976454734802), ('crap', 0.7620981335639954), ('flick', 0.7457514405250549), ('reason', 0.7417923808097839), ('series', 0.7396316528320312), ('book', 0.7309730052947998)]
[('original', 0.9143882393836975), ('quality', 0.9016523957252502), ('storyline', 0.9005738496780396), ('moments', 0.8907697200775146), ('short', 0.884209394454956), ('exciting', 0.8834180235862732), ('animation', 0.8807663321495056), ('dialogue', 0.8800377249717712), ('elements', 0.8788599371910095), ('points', 0.8745812773704529)]
[('great', 0.9210627675056458), ('bad', 0.8968755602836609), ('funny', 0.8900598883628845), ('quite', 0.8782914876937866), ('nice', 0.8340063095092773), ('pretty', 0.8135146498680115), ('very', 0.8110578060150146), ('interesting', 0.8037008047103882), ('stupid', 0.7974013686180115), ('terrible', 0.7971122860908508)]
[('funny', 0.9686699509620667), ('qu

📚 Similarly to `most_similar` used on words directly, we can use [**`similar_by_vector`**](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.similar_by_vector) on vectors to do the same thing:

In [9]:
# YOUR CODE HERE
# Example vector: W2V('good') - W2V('bad')
vector = wv['good'] - wv['bad']

# Find words similar to the resulting vector
similar_words = wv.similar_by_vector(vector)
print(similar_words)

[('who', 0.5402792096138), ('man', 0.3547621965408325), ('young', 0.35010793805122375), ('played', 0.3288957476615906), ('woman', 0.32262298464775085), ('been', 0.30186355113983154), ('by', 0.28751957416534424), ('baldwin', 0.2872709631919861), ('girl', 0.27638572454452515), ('his', 0.2701127231121063)]


# Arithmetic on words

Now, let's perform some mathematical operations on words, i.e. on their vector representations!

As any learned word is represented as a vector, you can do basic arithmetic operations, such as:

$$W2V(good) - W2V(bad)$$

❓ **Question** ❓ Do this mathematical operation and print the result

In [10]:
# YOUR CODE HERE
result = wv['good'] - wv['bad']
print(result)

[-0.47466123 -0.03206002  0.2739201   0.03200614  0.11926115 -0.15343165
 -0.16075164  0.27054     0.10506856 -0.4536478  -0.08122755 -0.09678096
  0.5215027  -0.11803707 -0.04329756  0.34403574 -0.44549978  0.18215674
 -0.13465285  0.06248808 -0.2999335  -0.12711038 -0.01577687  0.15275277
 -0.02396107 -0.06438918 -0.09056437  0.51887965 -0.02706876 -0.3981321
 -0.28386825  0.08890255  0.01650485  0.07118639 -0.21806288 -0.23226869
  0.11657993 -0.18300268 -0.34405196  0.91915613 -0.33505142 -0.05946553
 -0.46354562  0.03460759  0.45419014 -0.46787816  0.6189844  -0.32711327
  0.08569902  0.45522887  0.19504628 -0.19517201 -0.36593956  0.00290701
 -0.02390821  0.52385277  0.20009574  0.07223475  0.25954995  0.7230062
  0.22959259  0.38500845 -0.274714   -0.30716747  0.17873621 -0.04182184
 -0.32959172 -0.16348396  0.66651773  0.26909906 -0.6523317  -0.5463084
  0.35489157  0.3411435  -0.29355627  0.14042974 -0.06371805 -0.12659067
  0.12598836  0.05708858  0.00779701 -0.08016439 -0.32

Now, imagine for a second that the following equality holds true:

$$W2V(good) - W2V(bad) = W2V(nice) - W2V(stupid)$$

which is equivalent to:

$$W2V(good) - W2V(bad) + W2V(stupid) = W2V(nice)$$

❓ **Question** ❓ Let's, just for fun (as it would be bold of us to think that this equality holds true ...), do the operation $W2V(good) - W2V(bad) + W2V(stupid)$ and store it in a `res` variable (which will be a vector of size 100 that you can print).

In [11]:
# YOUR CODE HERE
res = wv['good'] - wv['bad'] + wv['stupid']
print(res)

[-0.28059736 -0.07618937  0.28073665 -0.07135901  0.20610097 -0.67861056
 -0.40348786  0.73502064 -0.43264383 -0.72368455 -0.28207275 -0.6058144
  0.3407098   0.10786369  0.10875341  0.00637195 -0.0442816  -0.3472771
 -0.41882983 -0.83886105  0.172524   -0.00539419  0.80332744  0.03079969
 -0.17964092 -0.26411206 -0.4000761   0.3587557  -0.21423091 -0.27995187
  0.4091949   0.1834017   0.21002214 -0.16826877 -0.5796673   0.34157276
  0.17976765 -0.12163611 -0.3038207   0.18800199 -0.2973376  -0.52554584
 -0.37743697  0.5402318   0.5055126  -0.36346108  0.31828383 -0.58030283
 -0.15123662  0.6722075   0.39244518 -0.4728263  -0.50783694  0.12530892
 -0.09939478  0.76315993  0.3422591   0.01632381 -0.3202758   0.31176674
  0.14420521  0.3267354   0.08674985 -0.11160547 -0.495014    0.5591356
 -0.08464967 -0.08758501  0.2474837   0.34444916 -0.5689165  -0.1469765
  0.5997627   0.6074548   0.37978154  0.2584042  -0.04018098 -0.43567595
 -0.30181992  0.08869915  0.0289973  -0.26661724 -0.785

We said earlier, that for any vector it is possible to see the closest vectors in the embedding space.

❓ **Question** ❓ Look at the closest vectors of `res`

💡 _Hint_: `similar_by_vector`

In [12]:
# YOUR CODE HERE
closest_vectors = wv.similar_by_vector(res)
print(closest_vectors)

[('nice', 0.805273175239563), ('good', 0.767576277256012), ('deed', 0.7648229598999023), ('such', 0.749530553817749), ('potential', 0.7449359893798828), ('decent', 0.7387317419052124), ('tough', 0.7271735072135925), ('always', 0.7270431518554688), ('great', 0.720100998878479), ('josie', 0.7188912630081177)]


Incredible right! You can do arithmetic operations on words!

❓ **Question** ❓ You can try on arithmetic such as 

$$W2V(Boy) - W2V(Girl) = W2V(Man) - W2V(Woman)$$

or 

$$W2V(Queen) - W2V(King) = W2V(actress) - W2V(actor)$$

❗ **Remark** ❗ You will probably see that the results are not perfect. But don't forget that you trained your model on a very small corpus.

In [13]:
# YOUR CODE HERE
# Perform arithmetic operations on word vectors
result_1 = wv['boy'] - wv['girl'] + wv['woman']
result_2 = wv['queen'] - wv['king'] + wv['actor']

# Find the closest words to the resulting vectors
similar_words_1 = wv.similar_by_vector(result_1)
similar_words_2 = wv.similar_by_vector(result_2)

print("Closest words to 'boy - girl + woman':", similar_words_1)
print("Closest words to 'queen - king + actor':", similar_words_2)

Closest words to 'boy - girl + woman': [('friend', 0.9611149430274963), ('boy', 0.9479799866676331), ('affair', 0.9443061947822571), ('child', 0.9410057067871094), ('woman', 0.9386959075927734), ('wig', 0.9366944432258606), ('career', 0.9355360269546509), ('coffin', 0.9307120442390442), ('sister', 0.9274718761444092), ('emmy', 0.9271841645240784)]
Closest words to 'queen - king + actor': [('actor', 0.9757517576217651), ('performance', 0.8691608309745789), ('actress', 0.8661665916442871), ('role', 0.8654139041900635), ('job', 0.8394347429275513), ('character', 0.8088074326515198), ('guy', 0.7912322282791138), ('villain', 0.7801989316940308), ('man', 0.772824764251709), ('inexperienced', 0.7579660415649414)]


<u><i>Some notes about Word2Vec as an internal Neural Network</i></u>:

You might wonder where does this magic comes from (at quite a low price, you just ran a line of code on a very small corpus and it was trained within few minutes). The magic comes from the way Word2Vec is trained. The details are quite complex, but you can remember that Word2vec, in `word2vec = Word2Vec(sentences=X_train)`, actually trains a internal neural network (that you don't see).  

In a nutshell, this internal neural network predicts a word from the surroundings words in a sentences. Hence, it splits the original sentences, then for each split it chooses some words as inputs $X$ and a word as the output $y$ which it tries to predict, using the embedding space.

And as with any neural network, Word2Vec has some hyperparameters. Let's play with some of these. 

# Word2Vec hyperparameters

❓ **Question** ❓ The first important hyperparameter is the `vector_size` argument. It corresponds to the size of the embedding space. Learn a new `word2vec_2` model, still trained on the `X_train`, but with a smaller or higher `vector_size`.

Verify on some words that the embedding size is the one you chose.

In [14]:
# YOUR CODE HERE
# Train a new Word2Vec model with a different vector_size
word2vec_2 = Word2Vec(sentences=X_train, vector_size=50)  # Smaller embedding size

# Verify the embedding size for some words
wv_2 = word2vec_2.wv
print(f"Embedding size for 'movie': {len(wv_2['movie'])}")
print(f"Embedding size for 'good': {len(wv_2['good'])}")

Embedding size for 'movie': 50
Embedding size for 'good': 50


❓ **Question** ❓ Use the **`Word2Vec.wv.key_to_index`** attribute to display the size of the learned vocabulary. Compare it to the number of different words in `X_train`.

In [15]:
# YOUR CODE HERE
# Size of the learned vocabulary
vocab_size = len(wv.key_to_index)
print(f"Size of the learned vocabulary: {vocab_size}")

# Number of different words in X_train
unique_words = set(word for sentence in X_train for word in sentence)
num_unique_words = len(unique_words)
print(f"Number of different words in X_train: {num_unique_words}")

Size of the learned vocabulary: 8006
Number of different words in X_train: 30419


There is an important difference between the number of words in the train sentences and in the Word2Vec vocabulary, even though it has been trained on the train sentence set. The reasons comes from the second important hyperparameter of Word2Vec:  `min_count`. 

`min_count` is a integer that tells you how many occurrences a given word should have to be learned in the embedding space. For instance, let's say that the word "movie" appears 1000 times in the corpus and "simba" only 2 times. If `min_count=3`, the word "simba" will be skipped during the training.

The intention is to learn a representation of words that are sufficiently present in the corpus to have a robust embedded representation.

❓ **Question** ❓ Learn a new `word2vec_3` model with a `min_count` higher than 5 (which is the default value) and a `word2vec_4` with a `min_count` smaller than 5, and then, compare the size of the vocabulary for all the different word2vecs that you have trained (you can choose any `vector_size` you want).

In [16]:
# YOUR CODE HERE
# Train a new Word2Vec model with a higher min_count
word2vec_3 = Word2Vec(sentences=X_train, vector_size=100, min_count=10)

# Train another Word2Vec model with a smaller min_count
word2vec_4 = Word2Vec(sentences=X_train, vector_size=100, min_count=2)

# Compare the size of the vocabulary for all the trained Word2Vec models
vocab_size_3 = len(word2vec_3.wv.key_to_index)
vocab_size_4 = len(word2vec_4.wv.key_to_index)

print(f"Vocabulary size with min_count=10: {vocab_size_3}")
print(f"Vocabulary size with min_count=2: {vocab_size_4}")
print(f"Vocabulary size with default min_count=5: {vocab_size}")

Vocabulary size with min_count=10: 4503
Vocabulary size with min_count=2: 16729
Vocabulary size with default min_count=5: 8006


Remember that Word2Vec has an internal neural network that is optimized based on some predictions. These predictions actually correspond to predicting a word based on surrounding words. The surroundings words are in a `window` which corresponds to the number of words taken into account. And you can train the Word2Vec with different `window` sizes.

❓ **Question** ❓ Train a new `word2vec_5` model with a `window` different than previously (default is 5).

In [17]:
# YOUR CODE HERE
# Train a new Word2Vec model with a different window size
word2vec_5 = Word2Vec(sentences=X_train, vector_size=100, window=10)

# Verify the model
wv_5 = word2vec_5.wv
print(f"Vocabulary size with window=10: {len(wv_5.key_to_index)}")

Vocabulary size with window=10: 8006


The arguments you have seen (`vector_size`, `min_count` and `window`) are usually the ones that you should start playing with to get a better performance for your model.

But you can also look at other arguments in the [**📚 Documentation - gensim.models.word2vec.Text8Corpus**](https://radimrehurek.com/gensim/models/word2vec.html#gensim.models.word2vec.Text8Corpus)

# Convert our train and test set to RNN-ready datasets

Remember that `Word2Vec` is the first step to the overall process of feeding such a representation into a RNN, as shown here:

<img src="word2vec_representation.png" width="400px" />



Now, let's work on Step 2 by converting the training and test data into their vector representation to be ready to be fed in RNNs.

❓ **Question** ❓ Now, write a function that, given a sentence, returns a matrix that corresponds to the embedding of the full sentence, which means that you have to embed each word one after the other and concatenate the result to output a 2D matrix (make sure that your output is a NumPy array)

❗ **Remark** ❗ You will probably notice that some words you are trying to convert throw errors as they are said not to belong to the dictionary:

- For the <font color=orange>test</font> set, this is understandable: <font color=orange>some words were not</font> in the <font color=blue>train</font> set and thus, their <font color=orange>embedded representation is unknown</font>
- for the <font color=blue>train set</font>, due to `min_count` hyperparameter, not all the words have a vector representation.

In any case, just skip the missing words here.

In [18]:
import numpy as np

example = ['this', 'movie', 'is', 'the', 'worst', 'action', 'movie', 'ever']
example_missing_words = ['this', 'movie', 'is', 'laaaaaaaaaame']

def embed_sentence(word2vec, sentence):
    # Embed each word in the sentence if it exists in the vocabulary
    embedded_words = [word2vec.wv[word] for word in sentence if word in word2vec.wv]
    # Convert the list of embeddings to a NumPy array
    return np.array(embedded_words)

### Checks
embedded_sentence = embed_sentence(word2vec, example)
assert(type(embedded_sentence) == np.ndarray)
assert(embedded_sentence.shape == (8, 100))

embedded_sentence_missing_words = embed_sentence(word2vec, example_missing_words)
assert(type(embedded_sentence_missing_words) == np.ndarray)
assert(embedded_sentence_missing_words.shape == (3, 100))

❓ **Question** ❓ Write a function that, given a list of sentences (each sentence being a list of words/strings), returns a list of embedded sentences (each sentence is a matrix). Apply this function to the train and test sentences

💡 _Hint_: Use the previous function `embed_sentence`

In [20]:
def embedding(word2vec, sentences):
    # Embed each sentence using the embed_sentence function
    return [embed_sentence(word2vec, sentence) for sentence in sentences]

X_train_embedded = embedding(word2vec, X_train)
X_test_embedded = embedding(word2vec, X_test)


❓ **Question** ❓ In order to have ready-to-use data, do not forget to pad your sequences so you have tensors which can be divided into batches (of `batch_size`) during the optimization. Store the padded values in `X_train_pad` and `X_test_pad`. Do not forget the important arguments of the padding ;)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences # type: ignore

### YOUR CODE HERE
# Pad the sequences to ensure uniform length
X_train_pad = pad_sequences([sentence.flatten() for sentence in X_train_embedded],
                            padding='post',
                            dtype='float32',
                            value=0.0)

X_test_pad = pad_sequences([sentence.flatten() for sentence in X_test_embedded],
                           padding='post',
                           dtype='float32',
                           value=0.0)
assert(len(X_train_pad.shape) == 3)
assert(len(X_test_pad.shape) == 3)
assert(X_train_pad.shape[2] == 100)
assert(X_test_pad.shape[2] == 100)



🏁 Congratulations, you are now able to use `Word2Vec` to embed your words :)

💾 Don't forget to git add/commit/push your notebook...

🚀 ... and move on to the next challenge!
